<a href="https://colab.research.google.com/github/saurabhkumar5/agentic_ai/blob/master/agentic_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 7: Agentic AI

# !pip install langchain langchain-openai tavily-python langchain-community
!pip install langgraph langchain-tavily

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["TAVILY_API_KEY"] = ""

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent

# LLM (Brain)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Tool (Search)
search_tool = TavilySearch(max_results=3)

# Tools list
tools = [search_tool]

# Agent banao (LangGraph version - naya tarika)
agent = create_react_agent(llm, tools)

/tmp/ipykernel_1442/3051448021.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [ ]:
result = agent.invoke({"messages": [("user", "What is the current weather in Berlin?")]})
print(result["messages"][-1].content)

The current weather in Berlin is 16°C with cloudy conditions. The humidity is at 78%, and the wind speed is 13 km/h.


In [ ]:
result = agent.invoke({"messages": [("user", "Who is the current Prime Minister of India and what is their age?")]})

for msg in result["messages"]:
    print(f"\n--- {msg.type} ---")
    print(msg.content)


--- human ---
Who is the current Prime Minister of India and what is their age?

--- ai ---


--- tool ---
{"query": "current Prime Minister of India 2023", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://en.wikipedia.org/wiki/Prime_Minister_of_India", "title": "Prime Minister of India", "content": "of P. V. Narasimha Rao, Atal Bihari Vajpayee, Manmohan Singh, and Narendra Modi, who is the current prime minister of India, serving since 26 May 2014. He is the first non-Congress leader to become PM after consecutive general elections and secure a third successive term (2014, 2019, 2024). The first prime minister to do so was Jawaharlal Nehru, who secured successive terms following the victory in the general elections of 1946, 1952, 1957, and 1962. [...] | |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  ---  ---  ---  ---  ---  ---  --- | |  | |  | | Prime Minister of India Narendra Modi | | | | | |  | |  | | | |  | |  | |  |  |  | |  | |  | |  | |  |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o", temperature=0)
search_tool = TavilySearch(max_results=3)
tools = [search_tool]

# System prompt mein current date bata do
agent = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 2026. Always search for the most current information.")

result = agent.invoke({"messages": [("user", "Who is the current Prime Minister of India and what is their age?")]})
print(result["messages"][-1].content)

/tmp/ipykernel_1442/182889996.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 2026. Always search for the most current information.")


The current Prime Minister of India as of September 2026 is Narendra Modi. 

To find his age, we need to know his birthdate. Narendra Modi was born on September 17, 1950. Therefore, as of September 2026, he is 76 years old.


In [ ]:
# multiple tools wala agent

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

llm = ChatOpenAI(model="gpt-4o", temperature=0)
search_tool = TavilySearch(max_results=3)

# Custom Calculator Tool
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression. Input should be a valid math expression like '2+2' or '100*5/3'"""
    try:
        return str(eval(expression))
    except:
        return "Invalid expression"

# Dono tools agent ko do
tools = [search_tool, calculator]

agent = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 2026.")

result = agent.invoke({"messages": [("user", "What is the population of India and if 30% of them are under 18, how many is that?")]})
print(result["messages"][-1].content)

/tmp/ipykernel_1442/1476778341.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 2026.")


The current population of India in 2026 is approximately 1,478,957,997. If 30% of them are under 18, that would be about 425,151,952 individuals.


In [ ]:
result = agent.invoke({"messages": [("user", "What is 1547 * 382 + 9021?")]})
print(result["messages"][-1].content)

The result of \(1547 \times 382 + 9021\) is 599,975.


In [ ]:
# Custom Tool banana

@tool
def days_between(dates: str) -> str:
    """Calculate days between two dates. Input format: 'YYYY-MM-DD to YYYY-MM-DD' like '2026-01-01 to 2026-12-31'"""
    from datetime import datetime
    try:
        date1, date2 = dates.split(" to ")
        d1 = datetime.strptime(date1.strip(), "%Y-%m-%d")
        d2 = datetime.strptime(date2.strip(), "%Y-%m-%d")
        diff = abs((d2 - d1).days)
        return f"{diff} days"
    except:
        return "Invalid format. Use: YYYY-MM-DD to YYYY-MM-DD"

# Ab 3 tools hain
tools = [search_tool, calculator, days_between]

agent = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 9, 2026.")

result = agent.invoke({"messages": [("user", "How many days until New Year 2027?")]})
print(result["messages"][-1].content)

/tmp/ipykernel_1442/1599893212.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 9, 2026.")


There are 114 days until New Year 2027.


In [ ]:
# Ab ek important cheez bachi hai — Memory in Agents. Abhi tak agent har sawaal ko alag se treat kar raha hai. Agar aap puchho "India ki population kya hai" phir puchho "uska 50% kitna hai" — agent ko nahi pata "uska" matlab kya hai kyunki pichla sawaal yaad nahi.

# Ye sikhein — agent mein memory kaise lagate hain?


# Pehle dekhte hain problem:

result = agent.invoke({"messages": [("user", "What is the population of Japan?")]})
print(result["messages"][-1].content)

As of 2026, the population of Japan is estimated to be approximately 122,427,731 people.


In [ ]:
# Ye run karo, phir iske baad ye run karo:

result = agent.invoke({"messages": [("user", "What is 50% of that?")]})
print(result["messages"][-1].content)

Could you please specify the number or value you want to find 50% of?


In [ ]:
# Ab memory lagate hain. Bahut simple hai:

from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent_with_memory = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 9, 2026.", checkpointer=memory)

/tmp/ipykernel_1442/2155498568.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_with_memory = create_react_agent(llm, tools, prompt="You are a helpful assistant. Today's date is September 9, 2026.", checkpointer=memory)


In [ ]:
config = {"configurable": {"thread_id": "chat-1"}}

result = agent_with_memory.invoke({"messages": [("user", "What is the population of Japan?")]}, config)
print(result["messages"][-1].content)

As of 2026, the population of Japan is estimated to be approximately 122,427,731 people.


In [ ]:
result = agent_with_memory.invoke({"messages": [("user", "What is 50% of that?")]}, config)
print(result["messages"][-1].content)

50% of Japan's population in 2026, which is approximately 122,427,731, is 61,213,865.5.


In [ ]:
# Aur ek cheez dekhte hain — agar naya thread_id do toh kya hoga:

config2 = {"configurable": {"thread_id": "chat-2"}}

result = agent_with_memory.invoke({"messages": [("user", "What is 50% of that?")]}, config2)
print(result["messages"][-1].content)

Could you please specify the number or value you want to find 50% of?


In [ ]:
# # Function Calling — Raw OpenAI API Se

# Pehle ek scenario samjho:

# Aap ek chatbot bana rahe ho jo weather bata sake. User bole "Delhi ka weather bata do" — toh LLM ko pata hona chahiye ki get_weather function call karna hai aur city "Delhi" dena hai.

# Step 1: Function define karo

import openai
import json

client = openai.OpenAI()

# Ye function actually weather dega (abhi dummy hai)
def get_weather(city):
    # Real mein yahan API call hogi
    weather_data = {
        "Delhi": "35°C, Sunny",
        "Berlin": "16°C, Cloudy",
        "London": "12°C, Rainy"
    }
    return weather_data.get(city, "Weather data not available")

# Ye LLM ko batata hai ki ye function available hai
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name like Delhi, Berlin, London"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [ ]:
# Step 2: LLM ko tools ke saath call karo
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "What is the weather in Delhi?"}],
    tools=tools
)

# Dekho LLM ne kya bola
message = response.choices[0].message
print("LLM ka response:")
print(f"Content: {message.content}")
print(f"Tool calls: {message.tool_calls}")

LLM ka response:
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_4jsFOnb0qyoKBrB0MqWjKeD1', function=Function(arguments='{"city":"Delhi"}', name='get_weather'), type='function')]


In [ ]:
# Step 3: Function actually chalao aur result LLM ko wapas do

# LLM ne jo function call maangi wo nikalo
tool_call = message.tool_calls[0]
function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print(f"Function: {function_name}")
print(f"Arguments: {arguments}")

# Ab actually function chalao
result = get_weather(arguments["city"])
print(f"Result: {result}")

# Result wapas LLM ko do taaki final answer banaye
final_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "What is the weather in Delhi?"},
        message,  # LLM ka pehla response (tool call wala)
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        }
    ],
    tools=tools
)

print(f"\nFinal Answer: {final_response.choices[0].message.content}")

Function: get_weather
Arguments: {'city': 'Delhi'}
Result: 35°C, Sunny

Final Answer: The current weather in Delhi is 35°C and sunny.


In [ ]:
# Ab ek interesting cheez dekhte hain — jab function call ki zarurat nahi hoti.
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
    tools=tools
)

message = response.choices[0].message
print(f"Content: {message.content}")
print(f"Tool calls: {message.tool_calls}")

Content: 2 + 2 equals 4.
Tool calls: None


In [ ]:
# Multiple Functions — LLM Khud Choose Karega

# Abhi weather function tha. Ab 2 aur functions add karte hain:

# Function 1: Weather (pehle se hai)
def get_weather(city):
    weather_data = {
        "Delhi": "35°C, Sunny",
        "Berlin": "16°C, Cloudy",
        "London": "12°C, Rainy"
    }
    return weather_data.get(city, "Weather data not available")

# Function 2: Population
def get_population(country):
    population_data = {
        "India": "1.44 billion",
        "Germany": "84 million",
        "Japan": "122 million"
    }
    return population_data.get(country, "Population data not available")

# Function 3: Currency converter
def convert_currency(amount, from_currency, to_currency):
    rates = {"USD_INR": 83.5, "EUR_INR": 91.2, "USD_EUR": 0.92}
    key = f"{from_currency}_{to_currency}"
    rate = rates.get(key, None)
    if rate:
        return f"{amount} {from_currency} = {amount * rate} {to_currency}"
    return "Conversion not available"

# Teeno functions ka menu card
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_population",
            "description": "Get the population of a country",
            "parameters": {
                "type": "object",
                "properties": {
                    "country": {"type": "string", "description": "The country name"}
                },
                "required": ["country"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert one currency to another",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount to convert"},
                    "from_currency": {"type": "string", "description": "Source currency like USD, EUR, INR"},
                    "to_currency": {"type": "string", "description": "Target currency like USD, EUR, INR"}
                },
                "required": ["amount", "from_currency", "to_currency"]
            }
        }
    }
]

In [ ]:
# Ab test karte hain — ek helper function banate hain taaki baar baar same code na likhna pade:

def ask(question):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": question}],
        tools=tools
    )
    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"Function chosen: {name}")
        print(f"Arguments: {args}")
    else:
        print(f"No function called. Direct answer: {message.content}")

# Test 1
ask("What is the weather in London?")
print("---")

# Test 2
ask("What is the population of Japan?")
print("---")

# Test 3
ask("Convert 100 USD to INR")
print("---")

# Test 4
ask("Who invented the light bulb?")

Function chosen: get_weather
Arguments: {'city': 'London'}
---
Function chosen: get_population
Arguments: {'country': 'Japan'}
---
Function chosen: convert_currency
Arguments: {'amount': 100, 'from_currency': 'USD', 'to_currency': 'INR'}
---
No function called. Direct answer: The invention of the light bulb is attributed to several inventors who made significant contributions to its development. However, Thomas Edison is most often credited with inventing the first practical and long-lasting electric light bulb in 1879. He improved upon previous designs by creating a vacuum inside the bulb and using a carbon filament, which greatly enhanced its efficiency and lifespan. Before Edison, Sir Joseph Swan in England had also developed a working electric light bulb using a carbonized paper filament around the same time. Both inventors were eventually recognized for their contributions and even formed a joint venture to produce and market electric light bulbs.


In [ ]:
# Multi-Agent System — Code Mein Banate Hain

# Scenario: User ek topic de — Researcher agent search kare, Writer agent blog post likhe. Supervisor decide kare kisko kaam dena hai.

# !pip install langgraph langchain-openai langchain-tavily

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0)
search_tool = TavilySearch(max_results=3)

# Researcher Agent - search karke data laaye
def researcher_agent(topic):
    print("--- RESEARCHER AGENT WORKING ---")
    search_results = search_tool.invoke(topic)

    response = llm.invoke([
        SystemMessage(content="You are a researcher. Summarize the search results into key points for a blog post. Give 4-5 bullet points only."),
        HumanMessage(content=f"Topic: {topic}\n\nSearch Results: {search_results}")
    ])
    print(f"Research done!\n")
    return response.content

# Writer Agent - research se blog post likhe
def writer_agent(research_data):
    print("--- WRITER AGENT WORKING ---")
    response = llm.invoke([
        SystemMessage(content="You are a blog writer. Write a short blog post (150-200 words) using the research points provided. Make it engaging and simple."),
        HumanMessage(content=f"Research Points:\n{research_data}")
    ])
    print(f"Blog written!\n")
    return response.content

# Supervisor - kaam baante
def supervisor(topic):
    print("=== SUPERVISOR: Starting task ===\n")

    # Step 1: Researcher ko kaam do
    print("SUPERVISOR: Sending task to Researcher...\n")
    research = researcher_agent(topic)
    print(f"Research Output:\n{research}\n")

    # Step 2: Writer ko research do
    print("SUPERVISOR: Sending research to Writer...\n")
    blog = writer_agent(research)
    print(f"Final Blog Post:\n{blog}\n")

    print("=== SUPERVISOR: Task complete! ===")
    return blog

In [ ]:
result = supervisor("AI trends in 2026")

=== SUPERVISOR: Starting task ===

SUPERVISOR: Sending task to Researcher...

--- RESEARCHER AGENT WORKING ---
Research done!

Research Output:
- **Deflation of the AI Bubble**: By 2026, the AI bubble is expected to deflate, potentially impacting the economy. This suggests a shift from hype-driven growth to more sustainable and realistic applications of AI.

- **Rise of Persistent Agents**: AI will see the growth of persistent agents, which are always-on assistants capable of handling long workflows and running locally to maintain data control. This trend emphasizes reliability and security in AI applications.

- **Shift in AI Research Priorities**: There will be a move towards robotics and physical AI, as the industry seeks new ideas beyond scaling large language models. This shift indicates a focus on tangible AI applications and innovation.

- **Multimodal and Open-Source AI**: The development of multimodal AI, which can interpret multisensory data, will become more prevalent. Addit

In [ ]:
# Lekin ek problem notice karo: Abhi ye supervisor "smart" nahi hai — usne hardcoded hai ki pehle researcher, phir writer. Asli multi-agent mein supervisor khud decide karta hai ki konse agent ko kaam dena hai, kab dena hai, kab wapas bhejna hai.

# Ab Reviewer Agent bhi add karte hain — jo blog check kare aur feedback de:

def reviewer_agent(blog):
    print("--- REVIEWER AGENT WORKING ---")
    response = llm.invoke([
        SystemMessage(content="You are a blog reviewer. Review the blog post and give specific feedback - is it engaging? Any factual issues? Any improvements needed? Keep feedback to 3-4 lines only."),
        HumanMessage(content=f"Blog Post:\n{blog}")
    ])
    print(f"Review done!\n")
    return response.content

# Updated Supervisor - ab 3 agents manage karta hai
def supervisor(topic):
    print("=== SUPERVISOR: Starting task ===\n")

    # Step 1: Researcher
    print("SUPERVISOR: Sending task to Researcher...\n")
    research = researcher_agent(topic)
    print(f"Research Output:\n{research}\n")

    # Step 2: Writer
    print("SUPERVISOR: Sending research to Writer...\n")
    blog = writer_agent(research)
    print(f"Blog Post:\n{blog}\n")

    # Step 3: Reviewer
    print("SUPERVISOR: Sending blog to Reviewer...\n")
    review = reviewer_agent(blog)
    print(f"Review Feedback:\n{review}\n")

    print("=== SUPERVISOR: Task complete! ===")
    return {"blog": blog, "review": review}

result = supervisor("Future of Robotics")

=== SUPERVISOR: Starting task ===

SUPERVISOR: Sending task to Researcher...

--- RESEARCHER AGENT WORKING ---
Research done!

Research Output:
- **Transformative Impact**: Robotics is reshaping various sectors, including construction, eldercare, healthcare, and manufacturing, by enhancing human capabilities and improving safety and precision.

- **Technological Advancements**: Breakthroughs in AI, haptics, and soft robotics are driving innovations, enabling robots to perform complex tasks, learn, adapt, and collaborate with humans more effectively.

- **Automation and Flexibility**: The future of robotics in manufacturing points towards fully automated production lines with robots capable of being reprogrammed for new tasks, increasing efficiency and reducing costs.

- **Challenges and Considerations**: As robotics technology advances, challenges such as global supply chain dependencies, workforce shifts, liability, and data security need strategic leadership and solutions to ensure s

In [ ]:
# Simple MCP Server — Code Kaisa Dikhta Hai

# Ek simple calculator MCP server banate hain:

# Pehle install karo:

!pip install mcp

In [ ]:
# (ye sirf samajhne ke liye hai, run nahi karenge abhi):

from mcp.server import FastMCP

# MCP Server banao
server = FastMCP("calculator-server")

# Tool 1: Add
@server.tool()
def add(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

# Tool 2: Multiply
@server.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

# Server start karo
server.run()

In [ ]:
# Agent MCP Server Se Kaise Connect Hota Hai — Client Side

# Ye code bhi samajhne ke liye hai:

from mcp.client import ClientSession
from mcp.client.stdio import stdio_client

# MCP Server se connect karo
async def main():
    # Server se connection banao
    async with stdio_client("calculator-server") as client:
        async with ClientSession(client) as session:

            # Step 1: Dekho server pe konse tools hain
            tools = await session.list_tools()
            print(f"Available tools: {tools}")

            # Step 2: Tool use karo
            result = await session.call_tool("add", {"a": 5, "b": 3})
            print(f"5 + 3 = {result}")

            result = await session.call_tool("multiply", {"a": 4, "b": 7})
            print(f"4 x 7 = {result}")